# LifeMate — GitHub → Colab → public submission
Run the cells in order. First choose **Runtime → Change runtime type → GPU**.
Only the repository URL and dataset ZIP path need editing. The code runs in its own Python 3.12 environment.
Your dataset stays in your private Google Drive; do not put it in GitHub.
The notebook uses base models, so you do not need teammate checkpoints yet.


In [ ]:
# 1. Edit this URL, then clone your code repository.
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git"
assert "YOUR_" not in REPO_URL, "Replace REPO_URL with your GitHub repository URL."
PROJECT = Path("/content/lifemate")
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
else:
    print("Using existing clone:", PROJECT)
os.chdir(PROJECT)
assert (PROJECT / "requirements.txt").is_file(), "Wrong repository or project root."
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
# 2. Mount your private Drive and locate the competition ZIP.
from google.colab import drive
drive.mount("/content/drive")
ARCHIVE = Path("/content/drive/MyDrive/train+public_test.zip")  # EDIT if stored elsewhere
assert ARCHIVE.is_file(), f"Upload the competition ZIP to Drive first: {ARCHIVE}"


In [ ]:
# 3. Extract the dataset onto Colab's local disk (faster than reading audio from Drive).
import shutil
DATA_DIR = Path("/content/lifemate-data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not list(DATA_DIR.glob("**/public_test/public_test.jsonl")):
    shutil.unpack_archive(str(ARCHIVE), str(DATA_DIR))
manifest_paths = list(DATA_DIR.glob("**/public_test/public_test.jsonl"))
assert len(manifest_paths) == 1, f"Expected one public_test manifest; found {manifest_paths}"
DATA_ROOT = manifest_paths[0].parent.parent
assert (DATA_ROOT / "train/train.jsonl").is_file(), "The ZIP must contain train and public_test."
print("Data root:", DATA_ROOT)


In [ ]:
# 4. Install a separate Python 3.12 environment; no Colab runtime restart needed.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = [sys.executable, "-m", "uv"]
ENV = PROJECT / ".venv-colab"
if not (ENV / "bin/python").is_file():
    subprocess.run(UV + ["venv", "--python", "3.12", str(ENV)], check=True)
PY = str(ENV / "bin/python")
subprocess.run(UV + ["pip", "install", "--python", PY, "torch==2.6.0+cu124",
                    "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run(UV + ["pip", "install", "--python", PY, "-r", "requirements.txt"], check=True)
subprocess.run([PY, "-c", "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.__version__, torch.cuda.get_device_name(0))"], check=True)


In [ ]:
# 5. Automatically download the main models. Public models: no HF token normally needed.
subprocess.run([PY, "scripts/download_models.py", "--profile", "baseline"], check=True)


In [ ]:
# 6. Configure paths and prepare the training-only examples used by the agent.
import json
cfg = json.loads((PROJECT / "configs/colab.json").read_text(encoding="utf-8-sig"))
cfg["data_root"] = str(DATA_ROOT)
CONFIG = PROJECT / "configs/colab-runtime.json"
CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
subprocess.run([PY, "-m", "baseline.prepare", "--data-root", str(DATA_ROOT)], check=True)
subprocess.run([PY, "-m", "baseline.doctor", "--config", str(CONFIG), "--split", "public_test"], check=True)


In [ ]:
# 7. Run ONE training audio on the Colab GPU before the full public run.
from datetime import datetime, timezone
RUN_DIR = Path("/content/drive/MyDrive/lifemate_runs") / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
RUN_DIR.mkdir(parents=True, exist_ok=False)
SMOKE = RUN_DIR / "colab_smoke.jsonl"
subprocess.run([PY, "-m", "baseline.run", "--config", str(CONFIG), "--split", "train",
                "--ids", "artifacts/splits/dev_ids.json", "--limit", "1", "--seconds", "120",
                "--output", str(SMOKE), "--verbose"], check=True)
subprocess.run([PY, "-m", "baseline.inspect_run", str(SMOKE), "--require-complete"], check=True)
print("Output directory (saved on Drive):", RUN_DIR)


## Full public run
Continue only after the one-sample run completes. The next cell processes the entire public split using the 28-second budget.
The final output and diagnostics are written directly to Drive. Leave Colab connected; the full run may take hours.
An interrupted run can leave a partial file. It must pass full-split validation before submission.
GPU availability and Colab runtime duration are not guaranteed. A completed smoke test does not establish accuracy or A30 performance.


In [ ]:
# 8. Generate ALL public_test predictions. No --limit here.
SUBMISSION = RUN_DIR / "submission.jsonl"
subprocess.run([PY, "-m", "baseline.run", "--config", str(CONFIG), "--split", "public_test",
                "--output", str(SUBMISSION)], check=True)


In [ ]:
# 9. Validate coverage/schema and review errors/timeouts.
subprocess.run([PY, "-m", "baseline.validate_submission", str(SUBMISSION),
                "--data-root", str(DATA_ROOT), "--split", "public_test"], check=True)
subprocess.run([PY, "-m", "baseline.inspect_run", str(SUBMISSION)], check=True)
print("Submit THIS file:", SUBMISSION)


In [ ]:
# 10. Download the result to your computer and upload it to the competition portal.
from google.colab import files
files.download(str(SUBMISSION))


## Later: teammate checkpoints
Upload checkpoints privately to Drive, copy them to Colab local disk, and change `cfg["asr"]["path"]` and `cfg["llm"]["adapter"]` in cell 6.
Use `TEAM_HANDOFF.md` for exact checkpoint formats. Start a NEW output directory for the new experiment.

If a Colab cell fails, fix that cell before proceeding. Installation/download/preparation cells may be rerun.
Prediction cells use exclusive output filenames: change the run directory for a new experiment. There is no automatic resume yet.

Sources: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [PyTorch builds](https://pytorch.org/get-started/previous-versions/).
